# Filipino SLM Training Experiment

This notebook sets up a controlled experiment to measure whether the Filipino Tokenizer
produces a better small language model than a generic tokenizer.

**Experiment design:**
- Same model architecture (GPT-2 mini, ~25M params)
- Same training data (Wikitext-TL-39)
- Same hyperparameters
- Different tokenizer → compare final **perplexity** on held-out Filipino text

**Hypothesis:** Lower perplexity with the Filipino Tokenizer, because:
1. Morpheme boundaries give the model linguistically meaningful token units
2. Higher vocabulary utilization — no wasted embedding rows
3. Lower fertility — more words fit in the context window

---

> ⚠️ **GPU required for actual training.**  
> This notebook runs fully on CPU up to the training cells.  
> Set `SKIP_TRAINING = False` on a GPU machine to run the full experiment.  
> Estimated time: ~2–4 hours on a single A100 / RTX 3090 for both tokenizers.

**Requirements:**
```bash
pip install filipino-tokenizer[hf] torch
```

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

SKIP_TRAINING = True   # Set False on a GPU machine to run full training

# Corpus path — update to match your environment
# Kaggle:  CORPUS = "/kaggle/input/<your-dataset>/train_corpus.txt"
# Local:   CORPUS = "../filipino_tokenizer/data/eval/train_corpus.txt"
CORPUS = "../filipino_tokenizer/data/eval/train_corpus.txt"

# Model
VOCAB_SIZE  = 32_000   # matches the bundled pretrained tokenizer
N_EMBD      = 384      # embedding dim  (GPT-2 small = 768)
N_LAYER     = 6        # transformer layers
N_HEAD      = 6        # attention heads
MAX_LENGTH  = 256      # max tokens per sample

# Training
TRAIN_SPLIT = 0.95     # fraction of corpus used for training
MAX_TRAIN_SAMPLES = 50_000   # cap for speed; remove for full run
BATCH_SIZE  = 16
LR          = 3e-4
EPOCHS      = 3
OUTPUT_DIR  = "models/slm_filipino"
BASELINE_DIR = "models/slm_gpt2tok"

print("Config loaded.")
print(f"  SKIP_TRAINING = {SKIP_TRAINING}")
print(f"  CORPUS        = {CORPUS}")
if SKIP_TRAINING:
    print("  Training cells will be skipped. Set SKIP_TRAINING = False on a GPU machine.")

In [ ]:
import os
import math
import torch

from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from filipino_tokenizer.tagalog import TagalogHFTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu" and not SKIP_TRAINING:
    print("WARNING: Training on CPU will be extremely slow. Use a GPU machine.")

## Step 1: Load the Filipino Tokenizer

`TagalogTokenizer.load_pretrained()` loads the 32k morphology-aware BPE model
bundled inside the `filipino-tokenizer` package — no path or download needed.

`TagalogHFTokenizer` then wraps it behind the HuggingFace `PreTrainedTokenizer`
interface so it plugs directly into `Trainer`.

In [ ]:
# Load the pretrained tokenizer bundled with the package — no path needed
fil_tokenizer = TagalogHFTokenizer()

# Save in HF format so Trainer can reload checkpoints
os.makedirs(OUTPUT_DIR, exist_ok=True)
fil_tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Filipino tokenizer loaded:")
print(f"  vocab_size : {fil_tokenizer.vocab_size:,}")
print(f"  bos_token  : {fil_tokenizer.bos_token!r} (id {fil_tokenizer.bos_token_id})")
print(f"  eos_token  : {fil_tokenizer.eos_token!r} (id {fil_tokenizer.eos_token_id})")
print(f"  pad_token  : {fil_tokenizer.pad_token!r} (id {fil_tokenizer.pad_token_id})")

## Step 2: Prepare the Dataset

We split Wikitext-TL-39 into train and validation sets, tokenize each split,
and wrap it in a HuggingFace `Dataset`.

In [ ]:
import re
from datasets import load_from_disk

FALLBACK_LINES = [
    "Kumain siya ng pagkain sa hapagkainan.",
    "Ang mga bata ay masayang naglalaro sa labas ng bahay.",
    "Nagtatrabaho ang tatay sa opisina araw-araw.",
    "Pinakamahusay ang ginawa niya sa pagtatanghal.",
    "Inaprubahan ng Senado ang panukalang batas kahapon.",
    "Nakapagpapakain na ang nanay ng maraming bisita.",
    "Bumili ako ng mga prutas sa palengke kahapon.",
    "Maganda ang panahon ngayon kaya lumabas kami para maglakad.",
    "Nagbabasa ang mga estudyante ng libro sa silid-aklatan.",
    "Nagluluto ang nanay ng masarap na adobo para sa buong pamilya.",
] * 100

if os.path.isfile(CORPUS):
    with open(CORPUS, "r", encoding="utf-8") as f:
        all_lines = [l.strip() for l in f if l.strip()]
    print(f"Corpus: {len(all_lines):,} lines from Wikitext-TL-39")
else:
    all_lines = FALLBACK_LINES
    print(f"Corpus not found. Using {len(all_lines)} fallback lines.")

if MAX_TRAIN_SAMPLES and len(all_lines) > MAX_TRAIN_SAMPLES:
    all_lines = all_lines[:MAX_TRAIN_SAMPLES]
    print(f"Capped to {MAX_TRAIN_SAMPLES:,} lines")

split_idx   = int(len(all_lines) * TRAIN_SPLIT)
train_lines = all_lines[:split_idx]
val_lines   = all_lines[split_idx:]
print(f"Train: {len(train_lines):,} lines  |  Val: {len(val_lines):,} lines")


def _prewarm(tokenizer, lines):
    """
    Pre-segment all unique words to warm the segmentation cache.
    Defined inline for compatibility with older installed versions of
    filipino-tokenizer that don't have TagalogTokenizer.prewarm() yet.
    Falls back to the method if available.
    """
    inner = tokenizer._inner
    if hasattr(inner, "prewarm"):
        inner.prewarm(lines)
        return
    unique_words = set()
    for line in lines:
        for part in re.split(r'(\s+|[^\w])', line):
            if part and re.match(r'^\w+$', part):
                unique_words.add(part.lower())
    total = len(unique_words)
    print(f"Pre-warming cache: {total:,} unique words ...", end="\r")
    for word in unique_words:
        if word not in inner._segment_cache:
            inner._segment_cache[word] = inner._surface_annotate(word)
    print(f"Cache warmed: {total:,} unique words segmented.   ")


def tokenize_dataset(lines, tokenizer, batch_size=5000):
    """Tokenize in batches with progress reporting."""
    all_input_ids, all_attention_masks = [], []
    for i in range(0, len(lines), batch_size):
        batch = lines[i : i + batch_size]
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors=None,
        )
        all_input_ids.extend(enc["input_ids"])
        all_attention_masks.extend(enc["attention_mask"])
        pct = min(i + batch_size, len(lines)) / len(lines) * 100
        print(f"  {pct:5.1f}%  {min(i+batch_size, len(lines)):,}/{len(lines):,} lines", end="\r")
    print()
    return Dataset.from_dict({
        "input_ids":      all_input_ids,
        "attention_mask": all_attention_masks,
    })


FIL_TRAIN_CACHE = "tokenized_fil_train"
FIL_VAL_CACHE   = "tokenized_fil_val"

if os.path.isdir(FIL_TRAIN_CACHE) and os.path.isdir(FIL_VAL_CACHE):
    print("Loading cached Filipino-tokenized datasets from disk...")
    train_ds = load_from_disk(FIL_TRAIN_CACHE)
    val_ds   = load_from_disk(FIL_VAL_CACHE)
    print("Loaded from cache.")
else:
    print("Pre-warming segmentation cache...")
    _prewarm(fil_tokenizer, all_lines)

    print("Tokenizing train split...")
    train_ds = tokenize_dataset(train_lines, fil_tokenizer)
    print("Tokenizing val split...")
    val_ds   = tokenize_dataset(val_lines, fil_tokenizer)

    print("Saving to disk...")
    train_ds.save_to_disk(FIL_TRAIN_CACHE)
    val_ds.save_to_disk(FIL_VAL_CACHE)
    print("Saved. Future runs will skip tokenization.")

print(f"\nTrain dataset: {len(train_ds):,} samples")
print(f"Val   dataset: {len(val_ds):,} samples")

## Step 3: Model Architecture

We use a GPT-2 style architecture with a smaller config (~25M params) —
large enough to learn meaningful language patterns, small enough to train in hours.

The only tokenizer-specific setting is `vocab_size`.

In [ ]:
def build_model(tokenizer):
    config = GPT2Config(
        vocab_size     = tokenizer.vocab_size,
        n_embd         = N_EMBD,
        n_layer        = N_LAYER,
        n_head         = N_HEAD,
        n_positions    = MAX_LENGTH,
        n_ctx          = MAX_LENGTH,
        pad_token_id   = tokenizer.pad_token_id,
        bos_token_id   = tokenizer.bos_token_id,
        eos_token_id   = tokenizer.eos_token_id,
    )
    return GPT2LMHeadModel(config)


model = build_model(fil_tokenizer)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model architecture: GPT-2 mini")
print(f"  Layers          : {N_LAYER}")
print(f"  Heads           : {N_HEAD}")
print(f"  Embedding dim   : {N_EMBD}")
print(f"  Max length      : {MAX_LENGTH}")
print(f"  Vocab size      : {fil_tokenizer.vocab_size:,}")
print(f"  Total params    : {total_params / 1e6:.1f}M")
print(f"  Trainable       : {trainable / 1e6:.1f}M")

# Verify one forward pass (CPU is fine for this)
sample_batch = train_ds[:4]
input_ids   = torch.tensor(sample_batch["input_ids"])
labels      = input_ids.clone()

with torch.no_grad():
    out = model(input_ids=input_ids, labels=labels)

print(f"\nForward pass OK:")
print(f"  Loss            : {out.loss.item():.4f}")
print(f"  Expected ~       : {math.log(fil_tokenizer.vocab_size):.2f} (random init)")
print(f"  Logits shape    : {list(out.logits.shape)}")

## Step 4: Training

> ⚠️ **This cell requires a GPU. Set `SKIP_TRAINING = False` at the top to run it.**

We use HuggingFace `Trainer` for the training loop — it handles gradient accumulation,
checkpointing, and evaluation automatically.

In [ ]:
def train_model(model, tokenizer, train_ds, val_ds, output_dir):
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,   # causal LM, not masked LM
    )

    args = TrainingArguments(
        output_dir          = output_dir,
        num_train_epochs    = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        learning_rate       = LR,
        lr_scheduler_type   = "cosine",
        warmup_ratio        = 0.05,
        weight_decay        = 0.01,
        evaluation_strategy = "steps",
        eval_steps          = 500,
        save_steps          = 500,
        save_total_limit    = 2,
        load_best_model_at_end = True,
        fp16                = torch.cuda.is_available(),
        logging_steps       = 100,
        report_to           = "none",
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        data_collator   = data_collator,
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer


if not SKIP_TRAINING:
    print("Training Filipino-tokenized model...")
    trainer_fil = train_model(model, fil_tokenizer, train_ds, val_ds, OUTPUT_DIR)
    print(f"Saved to {OUTPUT_DIR}/")
else:
    print("SKIPPED — set SKIP_TRAINING = False to run on a GPU machine.")
    print(f"Training would run for {EPOCHS} epochs on {len(train_ds):,} samples.")

## Step 5: Evaluate Perplexity

Perplexity measures how well the model predicts held-out text.  
**Lower perplexity = better language model.**

$$\text{PPL} = \exp\left(\frac{1}{N} \sum_{i=1}^{N} -\log P(x_i \mid x_{<i})\right)$$

We compute perplexity on the validation split, which the model never saw during training.

In [ ]:
def evaluate_perplexity(model, dataset, device="cpu", batch_size=8):
    """Compute perplexity on a dataset."""
    model.eval()
    model.to(device)
    total_loss  = 0.0
    total_tokens = 0

    for i in range(0, len(dataset), batch_size):
        batch = dataset[i : i + batch_size]
        input_ids      = torch.tensor(batch["input_ids"]).to(device)
        attention_mask = torch.tensor(batch["attention_mask"]).to(device)
        labels         = input_ids.clone()
        # Mask padding tokens in loss
        labels[attention_mask == 0] = -100

        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        n_tokens = (labels != -100).sum().item()
        total_loss   += out.loss.item() * n_tokens
        total_tokens += n_tokens

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)


if not SKIP_TRAINING and os.path.isdir(OUTPUT_DIR):
    trained_model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR)
    ppl_fil = evaluate_perplexity(trained_model, val_ds, device=device)
    print(f"Filipino Tokenizer model — Perplexity: {ppl_fil:.2f}")
else:
    print("Perplexity evaluation skipped (no trained model).")
    print("Run with SKIP_TRAINING = False on a GPU machine, then re-run this cell.")
    ppl_fil = None

## Step 6: Baseline — GPT-2 Tokenizer

We repeat the exact same experiment with the GPT-2 tokenizer as a baseline.
Same model architecture, same data, same hyperparameters — only the tokenizer changes.

This is the controlled comparison that answers the core question.

In [ ]:
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

print(f"GPT-2 tokenizer vocab size: {gpt2_tokenizer.vocab_size:,}")

GPT2_TRAIN_CACHE = "tokenized_gpt2_train"
GPT2_VAL_CACHE   = "tokenized_gpt2_val"

if not SKIP_TRAINING:
    if os.path.isdir(GPT2_TRAIN_CACHE) and os.path.isdir(GPT2_VAL_CACHE):
        print("Loading cached GPT-2-tokenized datasets from disk...")
        from datasets import load_from_disk
        gpt2_train_ds = load_from_disk(GPT2_TRAIN_CACHE)
        gpt2_val_ds   = load_from_disk(GPT2_VAL_CACHE)
        print("Loaded from cache.")
    else:
        print("Tokenizing dataset with GPT-2 tokenizer...")
        gpt2_train_ds = tokenize_dataset(train_lines, gpt2_tokenizer)
        gpt2_val_ds   = tokenize_dataset(val_lines,   gpt2_tokenizer)
        print("Saving GPT-2 tokenized datasets to disk...")
        gpt2_train_ds.save_to_disk(GPT2_TRAIN_CACHE)
        gpt2_val_ds.save_to_disk(GPT2_VAL_CACHE)
        print("Saved.")

    print("Building GPT-2-tokenized model (same architecture, different vocab size)...")
    gpt2_model = build_model(gpt2_tokenizer)
    total_p = sum(p.numel() for p in gpt2_model.parameters())
    print(f"  Total params: {total_p / 1e6:.1f}M  "
          f"(larger embedding: {gpt2_tokenizer.vocab_size:,} × {N_EMBD})")

    print("Training GPT-2-tokenized model...")
    os.makedirs(BASELINE_DIR, exist_ok=True)
    trainer_gpt2 = train_model(gpt2_model, gpt2_tokenizer,
                               gpt2_train_ds, gpt2_val_ds, BASELINE_DIR)

    gpt2_trained = GPT2LMHeadModel.from_pretrained(BASELINE_DIR)
    ppl_gpt2 = evaluate_perplexity(gpt2_trained, gpt2_val_ds, device=device)
    print(f"\nGPT-2 Tokenizer model — Perplexity: {ppl_gpt2:.2f}")
else:
    ppl_gpt2 = None
    print("SKIPPED — set SKIP_TRAINING = False to run on a GPU machine.")

## Step 7: Results Comparison

In [ ]:
if ppl_fil is not None and ppl_gpt2 is not None:
    import plotly.graph_objects as go

    improvement = (ppl_gpt2 - ppl_fil) / ppl_gpt2 * 100

    print("=" * 50)
    print(f"{'Tokenizer':<25} {'Perplexity':>12}")
    print("-" * 50)
    print(f"{'Filipino Tokenizer':<25} {ppl_fil:>12.2f}")
    print(f"{'GPT-2 Tokenizer':<25} {ppl_gpt2:>12.2f}")
    print("-" * 50)
    winner = "Filipino Tokenizer" if ppl_fil < ppl_gpt2 else "GPT-2 Tokenizer"
    print(f"Winner: {winner}  ({abs(improvement):.1f}% {'lower' if ppl_fil < ppl_gpt2 else 'higher'} perplexity)")
    print("=" * 50)

    fig = go.Figure(go.Bar(
        x=["Filipino Tokenizer", "GPT-2 Tokenizer"],
        y=[ppl_fil, ppl_gpt2],
        marker_color=["#22c55e", "#6b7280"],
        text=[f"{ppl_fil:.2f}", f"{ppl_gpt2:.2f}"],
        textposition="outside",
    ))
    fig.update_layout(
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(family="Inter, sans-serif"),
        title="Perplexity on Wikitext-TL-39 validation set<br>"
              "<sup>Lower is better. Same model architecture, same data — only tokenizer differs.</sup>",
        yaxis_title="Perplexity (lower = better)",
        height=400,
    )
    fig.show()
else:
    print("Results not available — training was skipped.")
    print("Run with SKIP_TRAINING = False to get actual perplexity numbers.")

## Expected Results

Based on the tokenizer-level metrics in `slm_tokenizer_comparison.ipynb`,
we expect the Filipino Tokenizer to win on perplexity because:

| Factor | Filipino Tokenizer | GPT-2 Tokenizer |
|--------|-------------------|------------------|
| Fertility | Lower — more words per context window | Higher — fewer words fit in context |
| Vocab utilization | High — most vocab seen in Filipino | Low — ~40% vocab is never seen |
| Morpheme boundaries | Preserved by constraint | Arbitrary splits |
| Embedding efficiency | All 32k rows get gradient signal | Many rows receive no gradient |

**What to do with the results:**

- If Filipino Tokenizer wins → **publish the comparison** as evidence for the library's value
- If results are close → investigate fertility and boundary metrics more deeply
- Try increasing model size (more layers/dim) — morpheme-aware tokenization tends to help more as model capacity grows

**Next steps:**
- Upload the trained model to HuggingFace Hub
- Evaluate on downstream tasks (e.g. Filipino NLU benchmarks)
- Try with Cebuano once the language module is implemented

## Step 8: Training Curves

Loss curves show how both models learned over time.  
If the Filipino Tokenizer model drops faster, it's learning more efficiently from the same data.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def extract_logs(trainer):
    train_steps, train_loss = [], []
    eval_steps, eval_loss   = [], []
    for entry in trainer.state.log_history:
        if "loss" in entry:
            train_steps.append(entry["step"])
            train_loss.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(entry["step"])
            eval_loss.append(entry["eval_loss"])
    return train_steps, train_loss, eval_steps, eval_loss

if not SKIP_TRAINING:
    fil_tr_steps,  fil_tr_loss,  fil_ev_steps,  fil_ev_loss  = extract_logs(trainer_fil)
    gpt2_tr_steps, gpt2_tr_loss, gpt2_ev_steps, gpt2_ev_loss = extract_logs(trainer_gpt2)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Training Loss", "Validation Loss"),
    )

    # Training loss
    fig.add_trace(go.Scatter(x=fil_tr_steps,  y=fil_tr_loss,  name="Filipino Tok",
                             line=dict(color="#22c55e")), row=1, col=1)
    fig.add_trace(go.Scatter(x=gpt2_tr_steps, y=gpt2_tr_loss, name="GPT-2 Tok",
                             line=dict(color="#6b7280")), row=1, col=1)

    # Validation loss
    fig.add_trace(go.Scatter(x=fil_ev_steps,  y=fil_ev_loss,  name="Filipino Tok",
                             showlegend=False, line=dict(color="#22c55e")), row=1, col=2)
    fig.add_trace(go.Scatter(x=gpt2_ev_steps, y=gpt2_ev_loss, name="GPT-2 Tok",
                             showlegend=False, line=dict(color="#6b7280")), row=1, col=2)

    fig.update_layout(
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(family="Inter, sans-serif"),
        title="Training and Validation Loss — Filipino Tok vs GPT-2 Tok",
        height=400,
    )
    fig.update_yaxes(title_text="Loss")
    fig.update_xaxes(title_text="Steps")
    fig.show()
else:
    print("Training curves not available — set SKIP_TRAINING = False to generate them.")

## Step 9: Tokenizer Analysis

Side-by-side comparison of how each tokenizer represents the same Filipino text:
- **Fertility** — average tokens per word (lower = more compact)
- **Token length distribution** — how long sequences are after tokenization
- **Sequence utilization** — how much of the 256-token window is actually used

In [ ]:
import numpy as np

# Sample 2000 lines for analysis (fast on any machine)
sample_lines = val_lines[:2000]

def analyze_tokenizer(lines, tokenizer, name):
    total_tokens, total_words = 0, 0
    seq_lengths = []
    for line in lines:
        words = [w for w in line.split() if w]
        total_words += len(words)
        ids = tokenizer.encode(line, truncation=True, max_length=MAX_LENGTH)
        total_tokens += len(ids)
        seq_lengths.append(len(ids))
    fertility = total_tokens / max(total_words, 1)
    utilization = np.mean(seq_lengths) / MAX_LENGTH * 100
    return {
        "name": name,
        "fertility": fertility,
        "seq_lengths": seq_lengths,
        "utilization": utilization,
        "mean_len": np.mean(seq_lengths),
    }

print("Analyzing tokenizers on 2,000 val lines...")
fil_stats  = analyze_tokenizer(sample_lines, fil_tokenizer,  "Filipino Tok")
gpt2_stats = analyze_tokenizer(sample_lines, gpt2_tokenizer, "GPT-2 Tok")

print(f"\n{'Metric':<30} {'Filipino Tok':>15} {'GPT-2 Tok':>15}")
print("-" * 62)
print(f"{'Fertility (tokens/word)':<30} {fil_stats['fertility']:>15.2f} {gpt2_stats['fertility']:>15.2f}")
print(f"{'Mean sequence length':<30} {fil_stats['mean_len']:>15.1f} {gpt2_stats['mean_len']:>15.1f}")
print(f"{'Context window utilization':<30} {fil_stats['utilization']:>14.1f}% {gpt2_stats['utilization']:>14.1f}%")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Fertility (tokens/word)", "Sequence Length Distribution"),
)

# Fertility bar
fig.add_trace(go.Bar(
    x=["Filipino Tok", "GPT-2 Tok"],
    y=[fil_stats["fertility"], gpt2_stats["fertility"]],
    marker_color=["#22c55e", "#6b7280"],
    text=[f"{fil_stats['fertility']:.2f}", f"{gpt2_stats['fertility']:.2f}"],
    textposition="outside",
    showlegend=False,
), row=1, col=1)

# Sequence length histogram
fig.add_trace(go.Histogram(
    x=fil_stats["seq_lengths"], name="Filipino Tok",
    marker_color="#22c55e", opacity=0.7, nbinsx=40,
), row=1, col=2)
fig.add_trace(go.Histogram(
    x=gpt2_stats["seq_lengths"], name="GPT-2 Tok",
    marker_color="#6b7280", opacity=0.7, nbinsx=40,
), row=1, col=2)

fig.update_layout(
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    font=dict(family="Inter, sans-serif"),
    title="Tokenizer Comparison on Filipino Val Set",
    barmode="overlay",
    height=400,
)
fig.update_yaxes(title_text="Tokens/word", row=1, col=1)
fig.update_xaxes(title_text="Sequence length (tokens)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.show()

## Step 10: Inference — Text Generation

Generate Filipino text from both trained models given the same prompt.  
Qualitative comparison: which model produces more fluent, morphologically correct Filipino?

In [ ]:
PROMPTS = [
    "Kumain siya ng",
    "Ang mga bata ay",
    "Nagtatrabaho ang",
    "Pinakamahusay ang",
]

def generate(model, tokenizer, prompt, max_new_tokens=30):
    model.eval()
    model.to(device)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


if not SKIP_TRAINING:
    fil_model_loaded  = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR)
    gpt2_model_loaded = GPT2LMHeadModel.from_pretrained(BASELINE_DIR)

    print("=" * 70)
    print(f"{'PROMPT':<25} {'FILIPINO TOK OUTPUT':<23} {'GPT-2 TOK OUTPUT'}")
    print("=" * 70)
    for prompt in PROMPTS:
        fil_out  = generate(fil_model_loaded,  fil_tokenizer,  prompt)
        gpt2_out = generate(gpt2_model_loaded, gpt2_tokenizer, prompt)
        print(f"Prompt : {prompt}")
        print(f"  Filipino Tok : {fil_out}")
        print(f"  GPT-2 Tok   : {gpt2_out}")
        print("-" * 70)

    # Also show token-level view for one prompt
    print("\nToken-level view of the prompt:")
    p = PROMPTS[0]
    print(f"  Prompt          : {p!r}")
    print(f"  Filipino tokens : {fil_tokenizer.tokenize(p)}")
    print(f"  GPT-2 tokens    : {gpt2_tokenizer.tokenize(p)}")
else:
    print("Inference skipped — set SKIP_TRAINING = False and run training first.")
    print("\nToken-level view (tokenizer comparison, no model needed):")
    for prompt in PROMPTS:
        print(f"\n  Prompt          : {prompt!r}")
        print(f"  Filipino tokens : {fil_tokenizer.tokenize(prompt)}")
        print(f"  GPT-2 tokens    : {gpt2_tokenizer.tokenize(prompt)}")